# Chapter 12. 모방학습 — 행동복제(BC)와 DAgger 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter12_1_behavior_cloning_dagger.ipynb)

책 본문: [12.1 모방학습: 행동 복제와 DAgger](https://smhanlab.com/book-ml/kor/ml2/chapter12/1.html)

3×4 회랑(중간 줄 두 칸 공사 중) 위에서 (1) 전문가 시연 5개로 행동복제 정책을
만들고 시연 밖 상태 (1,3)에서 왜 타임아웃되는지 확인하고, (2) DAgger 2라운드
가 그 문제를 어떻게 고치는지, (3) 15% 확률적 교란 환경에서 BC와 DAgger의
롤아웃 200회를 비교합니다. 마지막에 복합 오차 \(p^T\) 표를 직접 계산합니다.

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # 그림 저장 위치(책 본문의 SVG)

## 1. 3×4 회랑 환경 만들기

행 0(상단 회랑) / 1(중간, (1,1)·(1,2) 공사 중) / 2(하단). 시작 (2,0), 목표 (0,3).
벽에 부딪히면 제자리. 전문가 정책은 위험칸을 피한 최단 경로(Dijkstra).

In [2]:
import heapq, random

R, C = 3, 4
DEAD = {(1, 1), (1, 2)}          # 중간 줄: 공사 중(진입 즉시 실패)
GOAL = (0, 3)
S0 = (2, 0)

def step(s, a):
    r, c = s
    n = {"up": (r-1, c), "down": (r+1, c), "left": (r, c-1), "right": (r, c+1)}[a]
    return n if 0 <= n[0] < R and 0 <= n[1] < C else s   # 벽 = 제자리

# 전문가: 위험칸을 피한 최단 경로 (Dijkstra, 목적지에서 역방향)
dist = {(r, c): None for r in range(R) for c in range(C)}
dist[GOAL] = 0
pq = [(0, GOAL)]
while pq:
    d, s = heapq.heappop(pq)
    if d > dist[s]:
        continue
    for a in ("up", "down", "left", "right"):
        t = step(s, a)
        if t not in DEAD and dist[t] is None:
            dist[t] = d + 1
            heapq.heappush(pq, (dist[t], t))

def expert(s):
    r, c = s
    if (r, c) == GOAL:
        return None
    opts = [a for a in ("up", "down", "left", "right")
            if step((r, c), a) not in DEAD and dist[step((r, c), a)] is not None]
    return min(opts, key=lambda a: dist[step((r, c), a)])

print("(1,3)의 최단거리 =", dist[(1, 3)], " | 전문가의 행동 =", expert((1, 3)))

(1,3)의 최단거리 = 1  | 전문가의 행동 = up


## 2. 행동복제: 시연 5개 + 조회 테이블 + 기본값 'right'

시연은 (2,0)에서 출발한 5스텝 직진 경로. BC 정책은 시연 상태를 그대로 기억하는
테이블이고, **시연에 없던 상태**에서는 시연의 다수결 행동 `right`를 기본값으로 쓴다.
(실전에서는 이 '기본값' 대신 1-NN/신경망 보간이 쓰이지만, 실패 메커니즘은 동일하다.)

In [3]:
# 전문가 시연: S0에서 전문가 정책을 따라갈 뿐
demo_path, s = [S0], S0
while s != GOAL:
    s = step(s, expert(s))
    demo_path.append(s)
demo = list(zip(demo_path[:-1], [expert(x) for x in demo_path[:-1]]))
print("시연 (s, a):", demo)
print("시연 경로:", " -> ".join(map(str, demo_path)))

table = dict(demo)
def bc(s):
    return table.get(s, "right")   # 시연에 없던 상태 -> 기본값

def rollout(policy, s0, max_steps=15):
    s, path = s0, [s0]
    for _ in range(max_steps):
        if s == GOAL:
            return path, "goal"
        if s in DEAD:
            return path, "crash"
        s = step(s, policy(s))
        path.append(s)
    return path, "timeout"

for start in (S0, (1, 3)):
    p, res = rollout(bc, start)
    print(f"BC 시작 {start}: {' -> '.join(map(str, p[:8]))}{' ...' if len(p) > 8 else ''} => {res} ({len(p)-1}스텝)")

시연 (s, a): [((2, 0), 'up'), ((1, 0), 'up'), ((0, 0), 'right'), ((0, 1), 'right'), ((0, 2), 'right')]
시연 경로: (2, 0) -> (1, 0) -> (0, 0) -> (0, 1) -> (0, 2) -> (0, 3)
BC 시작 (2, 0): (2, 0) -> (1, 0) -> (0, 0) -> (0, 1) -> (0, 2) -> (0, 3) => goal (5스텝)
BC 시작 (1, 3): (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) ... => timeout (15스텝)


시작 (2,0)에서는 시연 경로를 그대로 따라 **5스텝 목표**다. 그런데 시작 (1,3)은
시연에 없으므로 `right`(기본값)를 고르고, (1,3)에서 right는 벽 → 제자리 →
**타임아웃**. 시연의 *수량*이 아니라 시연이 *어떤 상태 분포* 위에 놓여 있는가가 문제다.

## 3. DAgger: 에이전트가 달리고, 전문가가 방문한 상태에 라벨만 답한다

매 라운드 (1) 에이전트는 *자신의* 정책으로 진행, (2) 방문한 각 상태에
전문가의 행동을 라벨로 추가, (3) 다시 학습(여기서는 테이블 갱신). 시작 (1,3).

In [4]:
D = dict(demo)
def pol_D(s):
    return D.get(s, "right")

for rnd in range(1, 4):
    s, path = (1, 3), [(1, 3)]
    labels = {}
    for _ in range(5):
        if s == GOAL:
            res = "goal"; break
        labels[s] = expert(s)               # 전문가가 방문한 상태에 라벨
        if s in DEAD:
            res = "crash"; break
        s = step(s, pol_D(s))               # 에이전트는 자신의 정책으로 진행
        path.append(s)
    else:
        res = "timeout"
    D.update(labels)
    print(f"라운드 {rnd}: {' -> '.join(map(str, path))} => {res} | 추가 라벨 {labels} | |D|={len(D)}")
    if res == "goal":
        break

print("\n최종 테이블:", sorted(D.items()))
p, res = rollout(pol_D, (1, 3))
print("DAgger 정책으로 (1,3)에서:", " -> ".join(map(str, p)), "=>", res)

라운드 1: (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) -> (1, 3) => timeout | 추가 라벨 {(1, 3): 'up'} | |D|=6
라운드 2: (1, 3) -> (0, 3) => goal | 추가 라벨 {(1, 3): 'up'} | |D|=6

최종 테이블: [((0, 0), 'right'), ((0, 1), 'right'), ((0, 2), 'right'), ((1, 0), 'up'), ((1, 3), 'up'), ((2, 0), 'up')]
DAgger 정책으로 (1,3)에서: (1, 3) -> (0, 3) => goal


라운드 1은 여전히 '잘못' 간다 — 기본값 `right`로 벽을 5번 친다. 그런데
**그 5번의 '잘못'이 데이터를 만든다**: 방문 상태 (1,3)에 전문가의 `up`이
추가되고, 라운드 2는 **1스텝 목표**다. 전문가가 (2,?)에서 시연을 1000개 더
하더라도 (1,3) 라벨은 절대 생기지 않는 것과 대비하라.

## 4. 교란 실험: 15% 확률로 명령이 다른 이동으로 바뀌면

*시연에 없던* 시작 (1,3)에서 200회 롤아웃(seed 0~199). BC는 시연 밖에서
기본값 right로 배회하다 위험칸에 밀려들 가능성이 크고, DAgger 정책은
(1,3)에서 바로 위로 탈출한다.

In [5]:
def step_noise(s, a, rng, p=0.15):
    if rng.random() < p:                       # 15%: 명령이 다른 이동으로 바뀜
        a = rng.choice([x for x in ("up", "down", "left", "right") if x != a])
    return step(s, a)

def rollout_noise(policy, s0, seed, max_steps=15):
    rng = random.Random(seed)
    s = s0
    for _ in range(max_steps):
        if s == GOAL:
            return "goal"
        if s in DEAD:
            return "crash"
        s = step_noise(s, policy(s), rng)
    return "timeout"

def study(policy, n=200):
    res = [rollout_noise(policy, (1, 3), sd) for sd in range(n)]
    return res.count("goal"), res.count("crash"), res.count("timeout")

bc_g, bc_c, bc_t = study(bc)
dg_g, dg_c, dg_t = study(pol_D)
print(f"BC     : 목표 {bc_g}회 ({bc_g/2}%), 위험칸 {bc_c}회 ({bc_c/2}%), 미도달 {bc_t}회 ({bc_t/2}%)")
print(f"DAgger : 목표 {dg_g}회 ({dg_g/2}%), 위험칸 {dg_c}회 ({dg_c/2}%), 미도달 {dg_t}회 ({dg_t/2}%)")

BC     : 목표 57회 (28.5%), 위험칸 74회 (37.0%), 미도달 69회 (34.5%)
DAgger : 목표 185회 (92.5%), 위험칸 13회 (6.5%), 미도달 2회 (1.0%)


In [6]:
import numpy as np
cats = ["목표 도달", "위험칸 진입", "미도달"]
bc = [bc_g/2, bc_c/2, bc_t/2]
dg = [dg_g/2, dg_c/2, dg_t/2]
x = np.arange(3); w = 0.36
fig, ax = plt.subplots(figsize=(7.5, 4.6))
b1 = ax.bar(x - w/2, bc, w, label="행동 복제(BC)\n5개 시연만 학습", color="#1971c2")
b2 = ax.bar(x + w/2, dg, w, label="DAgger (2라운드)", color="#2f9e44")
for bars in (b1, b2):
    for r in bars:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 1, f"{r.get_height():g}%",
                ha="center", fontsize=11)
ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=12)
ax.set_ylabel("200회 롤아웃 중 비율 (%)")
ax.set_title("교란 15%, 시작 (1,3)에서 200회 롤아웃 (seed 0~199)")
ax.set_ylim(0, 105); ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch12_1_perturbation_study.svg", bbox_inches="tight")
plt.show()

DAgger의 위험칸 진입이 0이 *아닌* 이유: 라운드는 매번 *5스텝 샘플링*이라
(1,3)에서 'left'로 직접 튕겨 (1,2)로 가는 단일 사건처럼 *방문되지 않은*
교란 경로가 남는다 — 방문 기반 라벨링의 본질적 한계다(본문 FAQ 참조).

## 5. 복합 오차를 숫자로: \(p^T\)

매 스텝 성공률 \(p\)일 때 \(T\)스텝 완주 확률. 12.3절 표(\(p = 0.98, 0.99, 0.995\),
\(T = 50, 100, 200, 300\))를 직접 계산하고, 완주 확률 50%인 길이를 구한다.

In [7]:
import math
for p in (0.98, 0.99, 0.995):
    row = [f"{p**T:.3f}" for T in (50, 100, 200, 300)]
    T50 = math.log(0.5) / math.log(p)
    print(f"p={p}: T=50:{row[0]}  T=100:{row[1]}  T=200:{row[2]}  T=300:{row[3]}   | 50% 완주 길이 T≈{T50:.0f}")

T = np.arange(1, 301)
fig, ax = plt.subplots(figsize=(7, 4.2))
for p, col in ((0.98, "#e03131"), (0.99, "#1971c2"), (0.995, "#2f9e44")):
    ax.plot(T, [p ** t for t in T], label=f"p={p}", color=col)
ax.set_xlabel("episode 길이 T"); ax.set_ylabel("완주 확률 $p^{T}$")
ax.set_title("매 스텝 성공률 p에 따른 완주 확률 p^T (12.3절 표)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

p=0.98: T=50:0.364  T=100:0.133  T=200:0.018  T=300:0.002   | 50% 완주 길이 T≈34
p=0.99: T=50:0.605  T=100:0.366  T=200:0.134  T=300:0.049   | 50% 완주 길이 T≈69
p=0.995: T=50:0.778  T=100:0.606  T=200:0.367  T=300:0.222   | 50% 완주 길이 T≈138


## 확장 실험 (권장)

- **시연 500개 추가가 안 되는 이유 확인**: (2,1)·(2,2)·(2,3)에서 시작하는
  시연을 500개 만들어 테이블에 합친 뒤 (1,3)에서 다시 `rollout(bc, (1,3))` —
  여전히 타임아웃이어야 한다.
- **교란 확률 sweep**: `p`를 0.05/0.15/0.30으로 바꿔 `study()`의 위험칸 비율
  추이를 확인 — 교란이 커질수록 BC의 시연 밖 분포 비중이 커진다.
- **DAgger 라운드 수**: 라운드를 1~10으로 늘려 위험칸 진입이 얼마나 줄어드는지
  추적(완전 0이 안 되는 이유를 본문 FAQ에서 설명).